# Handling Data Skew in PySpark Joins

This notebook demonstrates how to identify and handle data skew during joins.

We will compare two approaches:

1. Broadcast Join — when one side of the join is small
2. Salting — when both sides are large and a join key is heavily skewed

The salting implementation dynamically identifies skewed keys instead of
hard-coding a specific customer ID.

## 1. Create a Skewed Fact Dataset

We intentionally create a dataset where one customer has a disproportionately
large number of records.

This simulates a common data-engineering problem where a small number of
keys contain a very large percentage of the data.

In [3]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("salting").master("local[*]").getOrCreate()

In [4]:
from pyspark.sql import functions as F

fact_df = (
    spark.range(10_000_000)
    .withColumn(
        "customer_id",
        F.when(
            F.col("id") < 9_000_000,
            F.lit("C1")
        ).otherwise(
            F.concat(
                F.lit("C"),
                (F.col("id") % 9999 + 2).cast("string")
            )
        )
    )
    .withColumn(
        "amount",
        (F.rand(seed=42) * 1000).cast("decimal(10,2)")
    )
)

fact_df.cache()

DataFrame[id: bigint, customer_id: string, amount: decimal(10,2)]

## 2. Inspect the Data Distribution

Before optimizing the join, we need to understand whether the join key is
actually skewed.

We count the number of records for each customer.

In [5]:
key_counts = (
    fact_df
    .groupBy("customer_id")
    .count()
    .orderBy(F.desc("count"))
)

key_counts.show(10, False)

+-----------+-------+
|customer_id|count  |
+-----------+-------+
|C1         |9000000|
|C950       |101    |
|C922       |101    |
|C907       |101    |
|C985       |101    |
|C988       |101    |
|C919       |101    |
|C959       |101    |
|C968       |101    |
|C935       |101    |
+-----------+-------+
only showing top 10 rows



## 3. Create the Customer Dataset

The customer table contains one record per customer.

Unlike the fact table, the customer key is unique.

In [6]:
customer_df = (
    spark.range(10_000)
    .withColumn(
        "customer_id",
        F.concat(
            F.lit("C"),
            (F.col("id") + 1).cast("string")
        )
    )
    .withColumn(
        "customer_name",
        F.concat(
            F.lit("Customer_"),
            (F.col("id") + 1).cast("string")
        )
    )
    .drop("id")
)

customer_df.show(10)

+-----------+-------------+
|customer_id|customer_name|
+-----------+-------------+
|         C1|   Customer_1|
|         C2|   Customer_2|
|         C3|   Customer_3|
|         C4|   Customer_4|
|         C5|   Customer_5|
|         C6|   Customer_6|
|         C7|   Customer_7|
|         C8|   Customer_8|
|         C9|   Customer_9|
|        C10|  Customer_10|
+-----------+-------------+
only showing top 10 rows



## 4. Baseline Join

First, perform the join without applying any skew-handling technique.

This gives us the baseline implementation that we will compare against
the optimized approaches.

In [7]:
normal_join = (
    fact_df
    .join(
        customer_df,
        "customer_id",
        "inner"
    )
)

normal_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- BroadcastHashJoin Inner BuildRight (9)
      :- Filter (5)
      :  +- InMemoryTableScan (1)
      :        +- InMemoryRelation (2)
      :              +- * Project (4)
      :                 +- * Range (3)
      +- BroadcastExchange (8)
         +- Project (7)
            +- Range (6)


(1) InMemoryTableScan
Output [3]: [id#0L, customer_id#2, amount#5]
Arguments: [id#0L, customer_id#2, amount#5], [isnotnull(customer_id#2)]

(2) InMemoryRelation
Arguments: [id#0L, customer_id#2, amount#5], CachedRDDBuilder(org.apache.spark.sql.execution.columnar.DefaultCachedBatchSerializer@6694d7de,StorageLevel(disk, memory, deserialized, 1 replicas),*(1) Project [id#0L, CASE WHEN (id#0L < 9000000) THEN C1 ELSE concat(C, cast(((id#0L % 9999) + 2) as string)) END AS customer_id#2, cast((rand(42) * 1000.0) as decimal(10,2)) AS amount#5]
+- *(1) Range (0, 10000000, step=1, splits=8)
,None), [id#0L ASC NULLS FIRST]

(3) Range [codegen id :

In [8]:
normal_join.count()

10000000

## 5. Broadcast Join

The customer table contains only a small number of rows compared with the
fact table.

When one side of a join is small enough to fit safely in executor memory,
Spark can broadcast it instead of performing a large shuffle.

This is the first optimization we will demonstrate.

In [9]:
broadcast_join = (
    fact_df
    .join(
        F.broadcast(customer_df),
        "customer_id",
        "inner"
    )
)

broadcast_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- BroadcastHashJoin Inner BuildRight (9)
      :- Filter (5)
      :  +- InMemoryTableScan (1)
      :        +- InMemoryRelation (2)
      :              +- * Project (4)
      :                 +- * Range (3)
      +- BroadcastExchange (8)
         +- Project (7)
            +- Range (6)


(1) InMemoryTableScan
Output [3]: [id#0L, customer_id#2, amount#5]
Arguments: [id#0L, customer_id#2, amount#5], [isnotnull(customer_id#2)]

(2) InMemoryRelation
Arguments: [id#0L, customer_id#2, amount#5], CachedRDDBuilder(org.apache.spark.sql.execution.columnar.DefaultCachedBatchSerializer@6694d7de,StorageLevel(disk, memory, deserialized, 1 replicas),*(1) Project [id#0L, CASE WHEN (id#0L < 9000000) THEN C1 ELSE concat(C, cast(((id#0L % 9999) + 2) as string)) END AS customer_id#2, cast((rand(42) * 1000.0) as decimal(10,2)) AS amount#5]
+- *(1) Range (0, 10000000, step=1, splits=8)
,None), [id#0L ASC NULLS FIRST]

(3) Range [codegen id :

In [10]:
broadcast_join.count()

10000000

## 6. When Broadcast Is Not the Right Approach

Broadcasting is useful when one side of the join is small.

However, if both datasets are large, broadcasting one side may not be
practical.

For that situation, we need another technique for handling heavily skewed
join keys.

The next section demonstrates dynamic salting.

## 7. Dynamically Detect Skewed Keys

We do not assume that a particular customer is skewed.

Instead, we calculate the average number of records per key and identify
keys whose frequency is significantly higher than the average.

The threshold is an engineering parameter and should be tuned according
to the data and workload.

In [11]:
avg_count = (
    key_counts
    .agg(F.avg("count").alias("avg_count"))
    .first()["avg_count"]
)

SKEW_FACTOR = 10

skewed_keys = (
    key_counts
    .filter(F.col("count") > avg_count * SKEW_FACTOR)
    .select("customer_id")
)

skewed_keys.show()

+-----------+
|customer_id|
+-----------+
|         C1|
+-----------+



## 8. Add Salt to the Fact Table

For a detected skewed key, we create multiple temporary salt buckets.

The salt is generated deterministically from the fact record ID.

Normal keys keep salt = 0.

In [12]:
SALT_BUCKETS = 10

skewed_key_flag = (
    skewed_keys
    .withColumn("is_skewed", F.lit(1))
)

salted_fact = (
    fact_df
    .join(
        F.broadcast(skewed_key_flag),
        "customer_id",
        "left"
    )
    .withColumn(
        "salt",
        F.when(
            F.col("is_skewed") == 1,
            F.pmod(
                F.xxhash64("id"),
                F.lit(SALT_BUCKETS)
            )
        ).otherwise(F.lit(0))
    )
    .drop("is_skewed")
)

salted_fact.groupBy("customer_id", "salt") \
    .count() \
    .filter(F.col("customer_id") == "C1") \
    .orderBy("salt") \
    .show()

+-----------+----+------+
|customer_id|salt| count|
+-----------+----+------+
|         C1|   0|900515|
|         C1|   1|899344|
|         C1|   2|899178|
|         C1|   3|901878|
|         C1|   4|900380|
|         C1|   5|901035|
|         C1|   6|898994|
|         C1|   7|899812|
|         C1|   8|900446|
|         C1|   9|898418|
+-----------+----+------+



## 9. Create Matching Salt Buckets on the Customer Table

The customer table originally contains only one row for the skewed key.

For the join to work, the skewed customer row is temporarily replicated
across the same salt buckets.

This does not create new business customers.

It only creates temporary join keys.

In [13]:
salted_customer = (
    customer_df
    .join(
        F.broadcast(skewed_key_flag),
        "customer_id",
        "left"
    )
    .withColumn(
        "salt_values",
        F.when(
            F.col("is_skewed") == 1,
            F.sequence(
                F.lit(0),
                F.lit(SALT_BUCKETS - 1)
            )
        ).otherwise(
            F.array(F.lit(0))
        )
    )
    .withColumn(
        "salt",
        F.explode("salt_values")
    )
    .drop("is_skewed", "salt_values")
)

salted_customer.filter(
    F.col("customer_id") == "C1"
).show()

+-----------+-------------+----+
|customer_id|customer_name|salt|
+-----------+-------------+----+
|         C1|   Customer_1|   0|
|         C1|   Customer_1|   1|
|         C1|   Customer_1|   2|
|         C1|   Customer_1|   3|
|         C1|   Customer_1|   4|
|         C1|   Customer_1|   5|
|         C1|   Customer_1|   6|
|         C1|   Customer_1|   7|
|         C1|   Customer_1|   8|
|         C1|   Customer_1|   9|
+-----------+-------------+----+



## 10. Join Using the Salted Key

The join now uses both customer_id and salt.

This allows the previously skewed C1 records to be distributed across
multiple join keys.

In [14]:
salted_join = (
    salted_fact
    .join(
        salted_customer,
        ["customer_id", "salt"],
        "inner"
    )
)

salted_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (39)
+- Project (38)
   +- SortMergeJoin Inner (37)
      :- Sort (19)
      :  +- Exchange (18)
      :     +- Project (17)
      :        +- Filter (16)
      :           +- BroadcastHashJoin LeftOuter BuildRight (15)
      :              :- Filter (5)
      :              :  +- InMemoryTableScan (1)
      :              :        +- InMemoryRelation (2)
      :              :              +- * Project (4)
      :              :                 +- * Range (3)
      :              +- BroadcastExchange (14)
      :                 +- Project (13)
      :                    +- Filter (12)
      :                       +- HashAggregate (11)
      :                          +- Exchange (10)
      :                             +- HashAggregate (9)
      :                                +- Filter (8)
      :                                   +- InMemoryTableScan (6)
      :                                         +- InMemoryRelation (7)
      :          

In [15]:
salted_join.count()

10000000

## 11. Compare the Approaches

There is no single join optimization that should always be used.

The appropriate technique depends on the data.

### Broadcast Join
Use when one side of the join is small enough to broadcast safely.

### Salting
Use when the join contains severe key skew and broadcasting the other
dataset is not practical.

### Normal Join
Use when the key distribution is reasonably balanced and no special
optimization is required.

### Handling Data Skew in PySpark Joins

A practical PySpark demonstration of identifying and handling data skew during joins using Broadcast Join and Salting.

#### Problem

When a join key is highly skewed, a small number of keys can contain a very large portion of the data.

For example:

- `C1` → ~9 million records
- Other customers → relatively few records

This can create an uneven workload across Spark partitions.

#### What This Project Demonstrates

1. Creating a skewed dataset
2. Measuring key distribution
3. Performing a baseline join
4. Using Broadcast Join when one side is small
5. Dynamically detecting skewed keys
6. Applying salting to skewed keys
7. Replicating matching dimension keys across salt buckets
8. Joining using `(customer_id, salt)`
9. Comparing the approaches

#### Approach

##### 1. Baseline Join

The fact and customer datasets are joined directly using:

`customer_id`

This establishes the baseline before applying optimization.

2. Broadcast Join

When the customer dataset is small enough to fit safely in executor memory, it can be broadcast to avoid a large shuffle.

```python
fact_df.join(
    F.broadcast(customer_df),
    "customer_id",
    "inner"
)

3. Dynamic Skew Detection

The pipeline calculates key frequencies and identifies keys whose record count is significantly higher than the average.

The skewed key is not hard-coded.

4. Salting

For detected skewed keys, a temporary salt value is added to the fact records.

The corresponding customer record is replicated across the same salt buckets.

The final join uses:

customer_id + salt

This distributes the previously concentrated key across multiple join keys.

Decision
Situation	Approach
Data is reasonably balanced	Normal Join
One side is small	Broadcast Join
Severe skew and broadcasting is not practical	Salting
Key Takeaway

Join optimization should start with understanding the data distribution.

The appropriate technique depends on the characteristics of the datasets rather than applying the same optimization to every join.

Technologies
PySpark
Apache Spark
Databricks